### GPX sample test

In [ ]:
from climbing_performance.gpx import parse_gpx, extract_climbs
from climbing_performance.weather import fetch_hourly_weather, nearest_weather_sample

route = parse_gpx("la_redoute.gpx")

if route.start_time is None or route.end_time is None:
    raise ValueError(
        "This GPX file has no timestamps. "
        "Use assign_estimated_times() before fetching time-specific weather."
    )

print(f"Distance: {route.distance_m / 1000:.2f} km")
print(f"Ascent: {route.ascent_m:.0f} m")

if route.min_elevation_m is not None and route.max_elevation_m is not None:
    print(
        f"Elevation range: "
        f"{route.min_elevation_m:.0f}–{route.max_elevation_m:.0f} m"
    )

climbs = extract_climbs(route)

weather_window = route.weather_query_window(
    location="midpoint",
)

samples = fetch_hourly_weather(
    latitude=weather_window.latitude,
    longitude=weather_window.longitude,
    start_date=weather_window.start_date,
    end_date=weather_window.end_date,
)

sample = nearest_weather_sample(samples, route.start_time)

latitude_str, longitude_str = f"({weather_window.latitude:.2f}", f"{weather_window.longitude:.2f})"
print(f"Nearest weather sample to route start time at: lat, lon: {latitude_str}, {longitude_str}")
print(sample)

Distance: 1.50 km
Ascent: 153 m
Elevation range: 136–289 m
Nearest weather sample to route start time at: lat, lon: (50.49, 5.71)
WeatherSample(time=datetime.datetime(2026, 4, 26, 17, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))), temperature_c=15.4, relative_humidity_pct=51.0, wind_speed_m_s=3.28, wind_direction_deg=21.0, wind_gusts_m_s=7.6)


In [ ]:
if route.start_time is None or route.end_time is None:
    raise ValueError(...)

### Weather sample test

In [1]:
from datetime import datetime
from zoneinfo import ZoneInfo

from climbing_performance.weather import (
    fetch_hourly_weather,
    nearest_weather_sample,
    wind_to_components,
)

latitude = 45.9237
longitude = 6.8694

samples = fetch_hourly_weather(
    latitude=latitude,
    longitude=longitude,
    start_date="2026-04-18",
    end_date="2026-04-19",
)

# Mont Blanc / Chamonix area uses Europe/Paris local time
target = datetime(
    2026,
    4,
    18,
    15,
    30,
    tzinfo=ZoneInfo("Europe/Paris"),
)

weather = nearest_weather_sample(samples, target)

# Suppose rider heading is 210° true bearing
headwind, crosswind = wind_to_components(
    wind_speed_m_s=weather.wind_speed_m_s,
    wind_direction_deg=weather.wind_direction_deg,
    rider_heading_deg=210.0,
)

print(f"Nearest sample: {weather.time}")
print(f"Temperature: {weather.temperature_c:.1f} °C")
print(f"Relative humidity: {weather.relative_humidity_pct:.0f}%")
print(f"Wind speed: {weather.wind_speed_m_s:.2f} m/s")
print(f"Wind direction: {weather.wind_direction_deg:.0f}°")
print(f"Wind gusts: {weather.wind_gusts_m_s:.2f} m/s" if weather.wind_gusts_m_s is not None else "Wind gusts: unavailable")
print(f"Headwind component: {headwind:.2f} m/s")
print(f"Crosswind component: {crosswind:.2f} m/s")

Nearest sample: 2026-04-18 15:00:00+02:00
Temperature: 15.8 °C
Relative humidity: 39%
Wind speed: 0.64 m/s
Wind direction: 309°
Wind gusts: 6.00 m/s
Headwind component: -0.10 m/s
Crosswind component: 0.63 m/s


### GPX sample test

In [2]:
if route.start_time is None or route.end_time is None:
    raise ValueError(...)